### Truncation thresholds tutorial

This notebooks provides the code to visualize what modes are being truncated based on the input threshold.

In [ ]:
import numpy as np
import matplotlib.gridspec as gridspec
from lsst.ts.ofc import OFC, OFCData, SensitivityMatrix
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import colors, cm

%load_ext autoreload
%autoreload 2

%matplotlib inline

Initialize below the sensitivity matrix of interest. It returns the rescaled (normalized) sensitivity matrix. You will need to modify this cell if you want to use other detectors than the ComCam ones. You can also modify it by setting ofc_data.zn_selected, in case not all zernikes will be used. When trying different normalization weights, you will need to re-run this notebook. 

In [ ]:
#ofc_data = OFCData('lsst')
ofc_data = OFCData('lsst', config_dir='/home/r/roodman/u/LSST/packages/ts_config_mttcs/MTAOS/v8/ofc' )
dz_sensitivity_matrix = SensitivityMatrix(ofc_data)
sensor_name_list = [        
        "R00_SW0",
        "R04_SW0",
        "R40_SW0",
        "R44_SW0",
]

field_angles = [
    ofc_data.sample_points[sensor] for sensor in sensor_name_list
]
print('Field Angles = ',field_angles)  # these are the 4 corners

dz_sensitivity_matrix = SensitivityMatrix(ofc_data)
sensitivity_matrix = dz_sensitivity_matrix.evaluate(
    field_angles, 0.0
)
print('Sensitivity matrix, evaluated at corners, shape = ',sensitivity_matrix.shape)


zn_idx = np.array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22])

print('Normal list of Zernikes, starting from z4 = ',zn_idx)
print('Normal list of DoFs ',ofc_data.dof_idx)

sensitivity_matrix = sensitivity_matrix[:, zn_idx, :]  #use zn_idx instead of ofc_data.zn_idx

size = sensitivity_matrix.shape[2]
print('Size to reshape = ',size)

sensitivity_matrix = sensitivity_matrix.reshape((-1, size))

sensitivity_matrix = sensitivity_matrix[..., ofc_data.dof_idx]
raw_sensitivity_matrix = sensitivity_matrix.copy()
print('Reshaped Smatrix = ',sensitivity_matrix.shape)

# so what is done here is to downsample to 19 Zernikes (from 25) and then concatinate the 4 corners to get a Zernike dimension of 76

normalization_matrix = np.diag(
    ofc_data.normalization_weights[ofc_data.dof_idx]
)

print('Shape of Normalization matrix =',normalization_matrix.shape)
print('Normalization vector in DOF = ',ofc_data.normalization_weights[ofc_data.dof_idx])

sensitivity_matrix = sensitivity_matrix @ normalization_matrix

## explore the Smatrix and the normalization matrix

In [ ]:
iz8 = 8 - 4
raw_sensitivity_matrix[iz8,1]  #M2 dx and (1,8)

In [ ]:
iz4 = 4- 4
jcamdz = 5  # Cam dz and (1,4)
print(raw_sensitivity_matrix[iz4,jcamdz])
print(raw_sensitivity_matrix[iz4+23,jcamdz])
print(raw_sensitivity_matrix[iz4+23*2,jcamdz])
print(raw_sensitivity_matrix[iz4+23*3,jcamdz])

print(sensitivity_matrix[iz4,jcamdz])
print(sensitivity_matrix[iz4+23,jcamdz])
print(sensitivity_matrix[iz4+23*2,jcamdz])
print(sensitivity_matrix[iz4+23*3,jcamdz])

In [ ]:
labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
         'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
         '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
         '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
         '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
         '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
         '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
         '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
         '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
         '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
         ]

In [ ]:
sensitivity_matrix.shape

In [ ]:
f,ax = plt.subplots(1,1)
nZs = 23
_ = ax.hist(sensitivity_matrix[0:nZs,:].flatten(),bins=25)
ax.set_yscale('log')
ax.set_xlabel('Sensitivity Matrix (w/ normalization) Coefficient magnitude')

In [ ]:
f,ax = plt.subplots(1,1,figsize=(12,6))
nZs = 23
r = ax.imshow(sensitivity_matrix[0:nZs,:])
tt = ax.set_xticks([0, 5, 10, 15, 20, 25, 30, 35, 40, 45], [labels[0], labels[5], labels[10], labels[15], labels[20], labels[25], labels[30], labels[35], labels[40],labels[45]])
tt = ax.set_yticks([0,5,10,15,20],['z4','z9','z14','z19','z24'])
f.colorbar(r)
f.suptitle('Sensitivity Matrix (w/ normalization) Coefficient magnitude')

In [ ]:
f,ax = plt.subplots(1,1,figsize=(12,6))
nZs = 23
r = ax.imshow(np.abs(sensitivity_matrix[0:nZs,:]),norm=colors.LogNorm())
tt = ax.set_xticks([0, 5, 10, 15, 20, 25, 30, 35, 40, 45], [labels[0], labels[5], labels[10], labels[15], labels[20], labels[25], labels[30], labels[35], labels[40],labels[45]])
f.colorbar(r)


In [ ]:
f,ax = plt.subplots(1,1,figsize=(12,6))
nZs = 23
r = ax.imshow(raw_sensitivity_matrix[0:nZs,:])
tt = ax.set_xticks([0, 5, 10, 15, 20, 25, 30, 35, 40, 45], [labels[0], labels[5], labels[10], labels[15], labels[20], labels[25], labels[30], labels[35], labels[40],labels[45]])
tt = ax.set_yticks([0,5,10,15,20],['z4','z9','z14','z19','z24'])
f.colorbar(r)
f.suptitle('Raw Sensitivity Matrix (w/o normalization) Coefficient magnitude')

In [ ]:
sensitivity_matrix[0,5]

In [ ]:
f,ax = plt.subplots(1,1,figsize=(12,6))
nZs = 23
r = ax.imshow(sensitivity_matrix[0:nZs,10:])
tt = ax.set_xticks([0, 5, 10, 15, 20, 25, 30, 35], [labels[10], labels[15], labels[20], labels[25], labels[30], labels[35], labels[40],labels[45]])
tt = ax.set_yticks([0,5,10,15,20],['z4','z9','z14','z19','z24'])
f.colorbar(r)
f.suptitle('Sensitivity Matrix (w/ normalization) Coefficient magnitude')

Below the characteristic modes given the normalization weights used are plotted on the left. On the right, is the singular values for each of them along with the cutoff. One can set the cutoff by number or by value.

In [ ]:
labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
         'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
         '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
         '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
         '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
         '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
         '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
         '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
         '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
         '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
         ]
u, s, vh = np.linalg.svd(sensitivity_matrix[:,  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]], full_matrices=True)
v = vh.T

s_use10 = s
v_use10 = v
fig = plt.figure(figsize=(9, 4), dpi=600)
gs = gridspec.GridSpec(1,2, width_ratios=[1.2,1])
a0 = plt.subplot(gs[0])
a1 = plt.subplot(gs[1])

r = a0.imshow(v, cmap = 'seismic', vmin = -1, vmax = 1)

a0.set_yticks([0, 5, 10], [labels[0], labels[5], labels[10]])
a0.set_xticks([0, 10], ['$v_0$', '$v_{10}$'])
fig.colorbar(r)
a0.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks
a0.xaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.yaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.set_xlabel('Characteristic modes')
a0.grid(alpha=0.25)
a0.tick_params(direction = 'in', labelsize=12)


a1.semilogy(s, '.-', markersize=4)
#truncate_index = 25
#cutoff =  0.85 * s[truncate_index - 1] / np.max(s)
#print(cutoff)
cutoff = 1.e-4
a1.semilogy(np.max(s)*cutoff*np.ones(45))
a1.set_xticks([0, 10], ['$\sigma_0$', '$\sigma_{10}$'])
a1.set_xlabel('Singular values')
a1.grid(alpha=0.25)
a1.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks

a1.xaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto set minor ticks on x-axis

fig.savefig('aos_charmodes_use10.png')

# try with the usual 22 DOFs 

In [ ]:
listOfDofs = list(range(0,16+1)) + list(range(30,34+1))
print(listOfDofs)

In [ ]:
labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
         'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
         '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
         '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
         '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
         '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
         '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
         '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
         '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
         '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
         ]

listOfDofs = list(range(0,14+1)) + list(range(30,34+1))
u, s, vh = np.linalg.svd(sensitivity_matrix[:,  listOfDofs], full_matrices=True)
v = vh.T

s_use20 = s
v_use20 = v

fig = plt.figure(figsize=(9, 4), dpi=600)
gs = gridspec.GridSpec(1,2, width_ratios=[1.2,1])
a0 = plt.subplot(gs[0])
a1 = plt.subplot(gs[1])

r = a0.imshow(v, cmap = 'seismic', vmin = -1, vmax = 1)

a0.set_yticks([0, 5, 10], [labels[0], labels[5], labels[10]])
a0.set_xticks([0, 10], ['$v_0$', '$v_{10}$'])
fig.colorbar(r)
a0.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks
a0.xaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.yaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.set_xlabel('Characteristic modes')
a0.grid(alpha=0.25)
a0.tick_params(direction = 'in', labelsize=12)


a1.semilogy(s, '.-', markersize=4)
#truncate_index = 25
#cutoff =  0.85 * s[truncate_index - 1] / np.max(s)
#print(cutoff)
cutoff = 1.e-4
a1.semilogy(np.max(s)*cutoff*np.ones(45))
a1.set_xticks([0, 10], ['$\sigma_0$', '$\sigma_{10}$'])
a1.set_xlabel('Singular values')
a1.grid(alpha=0.25)
a1.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks

a1.xaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto set minor ticks on x-axis

fig.savefig('aos_charmodes_use20dof.png')

In [ ]:
v[:,4]

In [ ]:
v[:,9]

# use 30 DOFs

In [ ]:
labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
         'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
         '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
         '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
         '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
         '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
         '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
         '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
         '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
         '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
         ]

listOfDofs = list(range(0,19+1)) + list(range(30,39+1))
u, s, vh = np.linalg.svd(sensitivity_matrix[:,  listOfDofs], full_matrices=True)
v = vh.T

s_use30 = s
v_use30 = v

fig = plt.figure(figsize=(9, 4), dpi=600)
gs = gridspec.GridSpec(1,2, width_ratios=[1.2,1])
a0 = plt.subplot(gs[0])
a1 = plt.subplot(gs[1])

r = a0.imshow(v, cmap = 'seismic', vmin = -1, vmax = 1)

a0.set_yticks([0, 5, 10], [labels[0], labels[5], labels[10]])
a0.set_xticks([0, 10], ['$v_0$', '$v_{10}$'])
fig.colorbar(r)
a0.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks
a0.xaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.yaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.set_xlabel('Characteristic modes')
a0.grid(alpha=0.25)
a0.tick_params(direction = 'in', labelsize=12)


a1.semilogy(s, '.-', markersize=4)
#truncate_index = 25
#cutoff =  0.85 * s[truncate_index - 1] / np.max(s)
#print(cutoff)
cutoff = 1.e-4
a1.semilogy(np.max(s)*cutoff*np.ones(45))
a1.set_xticks([0, 10], ['$\sigma_0$', '$\sigma_{10}$'])
a1.set_xlabel('Singular values')
a1.grid(alpha=0.25)
a1.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks

a1.xaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto set minor ticks on x-axis

fig.savefig('aos_charmodes_use30dof.png')

# Use all 50 DOFs

In [ ]:
labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
         'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
         '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
         '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
         '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
         '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
         '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
         '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
         '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
         '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
         ]
listOfCharModes = np.arange(50)
u, s, vh = np.linalg.svd(sensitivity_matrix[:,  listOfCharModes ], full_matrices=True)
v = vh.T

s_use50 = s
v_use50 = v

fig = plt.figure(figsize=(9, 4), dpi=600)
gs = gridspec.GridSpec(1,2, width_ratios=[1.2,1])
a0 = plt.subplot(gs[0])
a1 = plt.subplot(gs[1])

r = a0.imshow(v, cmap = 'seismic', vmin = -1, vmax = 1)

a0.set_yticks([0, 5, 10], [labels[0], labels[5], labels[10]])
a0.set_xticks([0, 10], ['$v_0$', '$v_{10}$'])
fig.colorbar(r)
a0.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks
a0.xaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.yaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.set_xlabel('Characteristic modes')
a0.grid(alpha=0.25)
a0.tick_params(direction = 'in', labelsize=12)


a1.semilogy(s, '.-', markersize=4)
#truncate_index = 25
#cutoff =  0.85 * s[truncate_index - 1] / np.max(s)
#print(cutoff)
cutoff = 1.e-4
a1.semilogy(np.max(s)*cutoff*np.ones(45))
a1.set_xticks([0, 10], ['$\sigma_0$', '$\sigma_{10}$'])
a1.set_xlabel('Singular values')
a1.grid(alpha=0.25)
a1.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks

a1.xaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto set minor ticks on x-axis
fig.savefig('aos_charmodes_use50dof.png')

## calculate the Zernike vector corresponding to a CMode

In [ ]:
sensitivity_matrix.shape

In [ ]:
kCmode = 0
cmode_vec = v[:,kCmode]
print(cmode_vec)

In [ ]:
zhit_vec = sensitivity_matrix @ cmode_vec
print(zhit_vec.shape)
print(zhit_vec)

In [ ]:
# compare Singular Values

f,ax = plt.subplots(1,1)
ax.semilogy(s_use10, '.-', markersize=4,label='Use 10 DOFs')

ax.semilogy(s_use20, '.-', markersize=4,label='Use 20 DOFs')
ax.semilogy(s_use30, '.-', markersize=4,label='Use 30 DOFs')

ax.semilogy(s_use50, '.-', markersize=4,label='Use 50 DOFs')
ax.set_xlabel('Singular values')
ax.legend()
f.savefig('aos_compare_nmodes.png')

# What Characteristic Modes have appreciable Z7 ?

In [ ]:
listOfDofs10 = list(range(0,9+1))
listOfDofs20 = list(range(0,14+1)) + list(range(30,34+1))


In [ ]:
iZ = 0 # assume this is Z4 ?
sensitivity_matrix[[iZ],  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]

In [ ]:
iZ = 1 # assume this is Z5 ?
sensitivity_matrix[[iZ],  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]

In [ ]:
iZ = 2 # assume this is Z6 ?
sensitivity_matrix[[iZ],  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]

In [ ]:
iZ = 3 # assume this is Z7 ?
print(sensitivity_matrix[[iZ],  listOfDofs10])

iZ = 3 # assume this is Z7 ?
print(sensitivity_matrix[[iZ],  listOfDofs20])

In [ ]:
iZ = 4 # assume this is Z8 ?
sensitivity_matrix[[iZ],  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]

In [ ]:
iZ = 7 # assume this is Z11 ?
sensitivity_matrix[[iZ],  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]